<p align="center">
   <img src="img/amazon2.jpg" width="1250" height="200"> 
</p>

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">


<h1 align="center"><b> 📈 Pronósticos de Ventas de Amazon </b></h1>

---

Los siguientes datos representan las ventas promedio diarias de un artefacto electrónico de Amazon. Se tienen 150 datos disponibles. El gerente de ventas está interesado en conocer cuánto serán las ventas diarias en la próxima semana dada la información disponible.

</div>

---

In [ ]:
# Instalar Librerías #
install.packages("stats")
install.packages("astsa")
install.packages("forecast")
install.packages("tseries")
install.packages("nortest")
install.packages("fBasics")
install.packages("car")
install.packages("MLmetrics")
install.packages("ggplot2")
install.packages("dplyr")
install.packages("lubridate")
install.packages("gridExtra")

In [ ]:
# Cargar Librerías #
library(stats)
library(astsa)
library(forecast)
library(tseries)
library(nortest)
library(fBasics)
library(car)
library(MLmetrics)
library(ggplot2)
library(dplyr)
library(lubridate)
library(gridExtra)

In [ ]:
# Carga de datos #
sales = c(10.01 ,10.07, 10.32	,9.75	,10.33  ,10.13	,10.36	,10.32	,10.13	,10.16	,10.58	,10.62	,10.86	,11.2,
          10.74 ,10.56, 10.48	,10.77	,11.33	,10.96	,11.16	,11.7	,11.39	,11.42	,11.94	,11.24	,11.59	,10.96,
          11.4	,11.02, 11.01	,11.23	,11.33	,10.83	,10.84	,11.14	,10.38	,10.9	,11.05	,11.11	,11.01	,11.22,
          11.21 ,11.91, 11.69	,10.93	,10.99	,11.01	,10.84	,10.76	,10.77	,10.88	,10.49	,10.5	,11	    ,10.98,
          10.61 ,10.48, 10.53	,11.07	,10.61	,10.86	,10.34	,10.78	,10.8	,10.33	,10.44	,10.5	,10.75	,10.4,
          10.40 ,10.34, 10.55   ,10.46	,10.82	,10.91	,10.87	,10.67	,11.11	,10.88	,11.28	,11.27	,11.44	,11.52,
          12.10 ,11.83, 12.62   ,12.41	,12.43	,12.73	,13.01	,12.74	,12.73	,12.76	,12.92	,12.64	,12.79	,13.05,
          12.69 ,13.01, 12.90   ,13.12  ,12.47	,12.47  ,12.94	,13.1	,12.91  ,13.39	,13.13	,13.34	,13.34	,13.14,
          13.49 ,13.87, 13.39   ,13.59  ,13.27  ,13.70	,13.2	,13.32	,13.15	,13.3	,12.94	,13.29	,13.26	,13.08,
          13.24	,13.31, 13.52   ,13.02  ,13.25  ,13.12	,13.26  ,13.11	,13.3	,13.06	,13.32	,13.1	,13.27	,13.64,
          13.58	,13.87, 13.53   ,13.41  ,13.25  ,13.50  ,13.58	,13.51	,13.77	,13.4)

In [ ]:
# Cantidad de Observaciones #
paste("Número de promedios diarios:",length(sales))
# Resumen de la serie de promedios diarios #
summary(sales)

In [ ]:
# Análisis de la serie de promedios diarios #
fecha <- seq(as.Date("2025-01-01"), by = "day", length.out = length(sales))

df_sales <- data.frame(
  fecha = fecha,
  sales = as.numeric(sales)
)
options(repr.plot.width=14, repr.plot.height=8)
ggplot(df_sales, aes(x = fecha, y = sales)) +
  geom_line(color = "darkgreen", size = 1) +
  labs(title = "Serie de Promedios Diarios", x = "Fecha", y = "Ventas") +
  theme_minimal()+
  theme(
    plot.title = element_text(hjust = 0.5, size = 18, face = "bold")
  )

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

<h1 align="center"><b> 📊 Homogeneidad de Varianzas </b></h1> 

### **Prueba de Levene**

En modelos **SARIMA**, la varianza debe ser constante. Si la varianza cambia con el tiempo o con el nivel de la serie, se aplica la **Transformación de Box-Cox** para estabilizarla.

**$H_0: \sigma_1^2=\sigma_2^2=...=\sigma_k^2 $ (Homocedasticidad) $\hspace{0.4cm}$ $vs$ $\hspace{0.4cm}$ $H_1: ∃_{i,j} \sigma_i^2\neq \sigma_j^2$ (Heterocedasticidad)**

---

1. **Detección (Prueba de Levene en la Serie Original):**
   * Se divide la serie en subperiodos y se aplica Levene.
   * **$p$-valor $< 0.05$:** Varianza no constante (heterocedasticidad) $\rightarrow$ **Se requiere Box-Cox**.

2. **Transformación (Box-Cox):**
   * Se ajusta la serie mediante $y^{(\lambda)} = \frac{y^\lambda - 1}{\lambda}$ (para $\lambda \neq 0$) o $\log(y)$ (para $\lambda = 0$) para fijar la varianza.

3. **Validación (SARIMA y Residuos):**
   * Se ajusta el modelo SARIMA a la serie transformada.
   * Se vuelve a aplicar Levene sobre los **residuos del SARIMA**.
   * **$p$-valor $\ge 0.05$:** Varianza de residuos estabilizada exitosamente.

4. **Conclusión:** 
   * Levene identifica si necesitas Box-Cox antes de modelar SARIMA, y confirma después si los residuos del modelo son verdaderamente homocedásticos.
   * Si el valor p resultante es menor que el nivel de significancia (por ejemplo, 0.05), se rechaza $H_0$, indicando que las varianzas no son homogéneas.

</div>   

In [ ]:
# División la serie en Ventanas #
segmentos <- cut(1:length(sales), breaks = 30)

# Prueba de Levene #
pl = leveneTest(sales ~ segmentos)
print(pl)

if (pl$`Pr(>F)`[1] < 0.05) {
  print("Rechazamos la hipótesis nula: Las varianzas son diferentes entre las ventanas.")
} else {
  print("No rechazamos la hipótesis nula: Las varianzas son iguales entre las ventanas.")
}

### **Gráficos de Box-Plot Dinámicos**

In [ ]:
# Estructuración de los datos con fechas #
fechas <- seq(as.Date("2024-01-01"), as.Date("2024-05-29"), by = "day")
datos <- data.frame(Fecha = fechas, Valor = sales)

In [ ]:
# Número de días por ventana
tamaño_ventana <- 30

In [ ]:
# Crear una columna que indique el número de la ventana
datos <- datos %>%
  mutate(Ventana = ceiling(as.numeric(Fecha - min(Fecha) + 1) / tamaño_ventana))

In [ ]:
# Crear el boxplot por ventanas
ggplot(datos, aes(x = as.factor(Ventana), y = Valor)) +
  geom_boxplot(fill = "skyblue", color = "darkblue") +
  labs(
    title = "Boxplot por Ventana de Ventas en AMAZON",
    x = "Ventana",
    y = "Sales"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(hjust = 0.5)
  )

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

### **Prueba de Estacionaridad**

La prueba de Dickey-Fuller es una prueba de raíz unitaria que se utiliza comúnmente para determinar si una serie de tiempo es estacionaria o no.     

$H_0$ : No Estacionaria $\hspace{0.4cm}$ $vs$  $\hspace{0.4cm}$ $H_a$ : Estacionaria

</div>

In [ ]:
# Aplicar la prueba de Dickey-Fuller #
print(adf.test(sales))
if (adf.test(sales)$p.value < 0.05) {
  print("La serie es estacionaria")
} else {
  print("La serie no es estacionaria")
}

### **Funciones de Autocorrelación y Autocorrelación Parcial**

In [ ]:
# Gráfica de la ACF y PACF #

# ACF
acf_data <- acf(sales, plot = FALSE)
acf_df <- with(acf_data, data.frame(lag = lag, acf = acf))

g1 <- ggplot(acf_df, aes(x = lag, y = acf)) +
  geom_bar(stat = "identity", fill = "steelblue") +
  geom_hline(yintercept = 0) +
  geom_hline(yintercept = c(-1.96 / sqrt(length(sales)), 1.96 / sqrt(length(sales))), 
             linetype = "dashed", color = "red") +
  labs(title = "Función de Autocorrelación", y = "ACF", x = "Rezago") +
  theme_minimal()+
  theme(
    plot.title = element_text(hjust = 0.5)
  )

# PACF
pacf_data <- pacf(sales, lag.max = 30, plot = FALSE)
pacf_df <- with(pacf_data, data.frame(lag = lag, pacf = acf))

g2 <- ggplot(pacf_df, aes(x = lag, y = pacf)) +
  geom_bar(stat = "identity", fill = "steelblue") +
  geom_hline(yintercept = 0) +
  geom_hline(yintercept = c(-1.96 / sqrt(length(sales)), 1.96 / sqrt(length(sales))), 
             linetype = "dashed", color = "red") +
  labs(title = "Función de Autocorrelación Parcial", y = "PACF", x = "Rezago") +
  theme_minimal()+
  theme(
    plot.title = element_text(hjust = 0.5)
  )

# Mostrar los dos gráficos juntos
grid.arrange(g1, g2, ncol = 1)

In [ ]:
# Primera diferencia de ventas #
dsales<-diff(sales)
# Cantidad de Observaciones #
paste("Número de promedios diarios:",length(dsales))
# Resumen de la serie de promedios diarios #
summary(dsales)

In [ ]:
# Análisis de la serie diferenciada de promedios diarios #

fecha <- seq(as.Date("2024-01-01"), by = "day", length.out = length(dsales))

df_sales <- data.frame(
  fecha = fecha,
  sales = as.numeric(dsales)
)
options(repr.plot.width=14, repr.plot.height=8)
ggplot(df_sales, aes(x = fecha, y = sales)) +
  geom_line(color = "darkgreen", size = 1) +
  labs(title = "Serie Diferenciada de Promedios Diarios", x = "Fecha", y = "Ventas") +
  theme_minimal()+
  theme(
    plot.title = element_text(hjust = 0.5, size = 18, face = "bold")
  )

In [ ]:
# Gráfica de la ACF y PACF de la serie diferenciada #
# ACF
acf_data <- acf(dsales, plot = FALSE)
acf_df <- with(acf_data, data.frame(lag = lag, acf = acf))

g1 <- ggplot(acf_df, aes(x = lag, y = acf)) +
  geom_bar(stat = "identity", fill = "steelblue") +
  geom_hline(yintercept = 0) +
  geom_hline(yintercept = c(-1.96 / sqrt(length(sales)), 1.96 / sqrt(length(sales))), 
             linetype = "dashed", color = "red") +
  labs(title = "Función de Autocorrelación de Serie Diferenciada", y = "ACF", x = "Rezago") +
  theme_minimal()

# PACF
pacf_data <- pacf(dsales, lag.max = 30, plot = FALSE)
pacf_df <- with(pacf_data, data.frame(lag = lag, pacf = acf))

g2 <- ggplot(pacf_df, aes(x = lag, y = pacf)) +
  geom_bar(stat = "identity", fill = "steelblue") +
  geom_hline(yintercept = 0) +
  geom_hline(yintercept = c(-1.96 / sqrt(length(sales)), 1.96 / sqrt(length(sales))), 
             linetype = "dashed", color = "red") +
  labs(title = "Función de Autocorrelación Parcial de Serie Diferenciada", y = "PACF", x = "Rezago") +
  theme_minimal()

# Mostrar los dos gráficos juntos
grid.arrange(g1, g2, ncol = 1)

## Búsqueda y selección del mejor modelo con la función ARIMA

In [ ]:
# Función que busca el mejor modelo según los criterios AIC Y BIC ##
sarima.list = function(datos, p, d = 0, q, P = 0, D = 0, Q = 0, S = NA, include.mean = F, criterio = ""){
  M <- matrix(ncol = 10,nrow = (p+1)*(q+1)*(P+1)*(Q+1),dimnames=list(NULL,c("p","d","q","P","D","Q","S","converge","AIC", "BIC")))
  k <- 1
  n <- length(datos)
  for(i in 0:p){
    for(j in 0:q){
      for(l in 0:P){
        for(m in 0:Q){
          if ((i==0)&&(j==0)&&(l==0)&&(m==0)) next #Continua con la siguiente iteracion
          fit <- arima(datos, order = c(i, d, j),seasonal = list(order = c(l, D, m), period = S), include.mean = include.mean)
          M[k,1]  <- i
          M[k,2]  <- d
          M[k,3]  <- j
          M[k,4]  <- l
          M[k,5]  <- D
          M[k,6]  <- m
          M[k,7]  <- S
          M[k,8]  <- fit$code  # 0: Convergencia, 1: No Convergencia
          M[k,9]  <- AIC(fit)  # AIC
          M[k,10] <- AIC(fit, k = log(length(datos)))  # BIC
          k <- k+1
        }
      }
    }
  }
  if(criterio == "AIC"){
    M <- M[order(M[,9]),]
  }
  if(criterio == "BIC"){
    M <- M[order(M[,10]),]
  }
  if(criterio == ""){
    M <- M
  }
  rownames(M) = rep("", (p+1)*(q+1)*(P+1)*(Q+1))
  return(M[1:((p+1)*(q+1)*(P+1)*(Q+1)-1),])
}

In [ ]:
# Aplicación de la función a la serie original "sales" #
sari = sarima.list(sales, p=3, d=1, q=3, P = 0, D = 0, Q = 0, S = NA, include.mean = F, criterio = "")
sari


### **Estimación de parámetros para un ARIMA(0,1,1)**

### **Usando la Función ARIMA()**

In [ ]:
# Estimación ML condicional #
(mod_1 <- arima(sales, c(0, 1, 1), method = c("CSS")))
res_1 <- residuals(mod_1)

In [ ]:
# Estimación ML exacta #
(mod_2 <- arima(sales, c(0, 1, 1), method = c("ML")))
res_2 <- residuals(mod_2)

In [ ]:
# Estimación ML exacta con valores iniciales dados por la estimación condicional #
(mod_3 <- arima(sales, c(0, 1, 1), method = c("CSS-ML")))
res_3 <- residuals(mod_3)

### Usando la Función SARIMA()

In [ ]:
mod_sar <- sarima(sales, p = 0, d = 1, q = 1)

In [ ]:
# Valores de la Función SARIMA #
names(mod_sar) # Valores del Objeto sarima()
print(mod_sar$ttable) # Significancia de los parámetros estimados
mod_sar$fit # Valores Estimados
mod_sar$ICs # Todos los Criterios de Información
mod_sar$ICs[1] # Criterio de Información Akaike
mod_sar$ICs[2] # Criterio de Información Akaike Corregido
mod_sar$ICs[3] # Criterio de Información de Schwarz Bayesiano

---
### **Diagnóstico Residual para Modelo ARIMA(0,1,1)**

In [ ]:
# Diagnóstico residual paso a paso #
res_sar <- mod_sar$fit$residuals # Residuales del modelo SARIMA
par(mfrow=c(3,2))
plot.ts(res_sar, main="Residuales", ylab="")
hist(res_sar, main="Histograma de Residuales")
acf(res_sar, lag.max=25, main="ACF de Residuales")
pacf(res_sar, lag.max=25, main="PACF de Residuales")

# La prueba de Ljung-Box se formula para evaluar la autocorrelación de un conjunto de rezagos en los residuos de un modelo ajustado.
# La hipótesis nula (H0) es que no hay autocorrelación en los residuos hasta el rezago especificado, mientras que
# la hipótesis alternativa (H1) es que hay autocorrelación en al menos uno de los rezagos.
Box.Test = function(x, lag = 25, main = "p values for Ljung-Box statistic"){
  B<-vector("numeric")
  for(i in 1:lag){
    B[i]<-Box.test(x, lag = i, type = "Ljung-Box")$p.value
  }
  A<-matrix(cbind(c(1:lag),B), nrow = lag, ncol = 2, byrow=F, dimnames = list(NULL, c("lag", "p.value")))
  plot(A[,1], A[,2], ylim = c(0, max(0.051,(max(A[,2])+.01))),
       ylab = "p-value", xlab = "Lag", main = main, lwd = 2)
  abline(0.05, 0, col = 4, lty = 2)
}
Box.Test(res_sar)
hist(res_sar, freq = FALSE, main="Distribución de los residuales")
dens <- density(res_sar)
lines(dens, col = "red", lwd = 2)

In [ ]:
# 1. Extraer residuales y configurar parámetros
res_sar <- as.numeric(mod_sar$fit$residuals)
n <- length(res_sar)
lags <- 25

# Intervalos de confianza al 95% (Fórmulas de Quenouille y Bartlett para ruido blanco)
ci_acf  <- qnorm((1 + 0.95) / 2) / sqrt(n)
ci_pacf <- qnorm((1 + 0.95) / 2) / sqrt(n) 

# 2. Generar datos para cada diagnóstico
# A. Serie Temporal
df_ts <- data.frame(
  Panel = "1. Serie de Residuales",
  x = seq_along(res_sar),
  y = res_sar,
  xend = NA, yend = NA, grupo = "Serie"
)

# B. ACF
acf_vals <- acf(res_sar, lag.max = lags, plot = FALSE)$acf[-1]
df_acf <- data.frame(
  Panel = "3. Autocorrelación (ACF)",
  x = 1:lags,
  y = acf_vals,
  xend = 1:lags, yend = 0, grupo = "ACF"
)

# C. PACF
pacf_vals <- pacf(res_sar, lag.max = lags, plot = FALSE)$acf
df_pacf <- data.frame(
  Panel = "4. Autocorrelación Parcial (PACF)",
  x = 1:lags,
  y = pacf_vals,
  xend = 1:lags, yend = 0, grupo = "PACF"
)

# Data frame para dibujar los intervalos en ACF y PACF simultáneamente
df_ci <- data.frame(
  Panel = rep(c("3. Autocorrelación (ACF)", "4. Autocorrelación Parcial (PACF)"), each = 2),
  ci = c(ci_acf, -ci_acf, ci_pacf, -ci_pacf)
)

# D. Ljung-Box
p_vals <- sapply(1:lags, function(k) Box.test(res_sar, lag = k, type = "Ljung-Box")$p.value)
df_lb <- data.frame(
  Panel = "5. p-valores: Ljung-Box",
  x = 1:lags,
  y = p_vals,
  xend = NA, yend = NA, grupo = ifelse(p_vals < 0.05, "Rechazo", "No Rechazo")
)

# E. Histograma y Densidades
df_hist <- data.frame(
  Panel = "2. Histograma y Densidad",
  x = res_sar, y = NA, xend = NA, yend = NA, grupo = "Hist"
)
x_seq <- seq(min(res_sar), max(res_sar), length.out = 100)
df_norm <- data.frame(
  Panel = "2. Histograma y Densidad",
  x = x_seq,
  y = dnorm(x_seq, mean = mean(res_sar), sd = sd(res_sar)),
  xend = NA, yend = NA, grupo = "Normal Teórica"
)

# 3. Construir el gráfico unificado con ggplot2
ggplot() +
  # Serie de residuales (Línea)
  geom_line(data = df_ts, aes(x = x, y = y), color = "steelblue") +
  geom_hline(data = df_ts, aes(yintercept = 0), linetype = "dashed", color = "darkred") +
  
  # ACF y PACF (Segmentos y Puntos)
  geom_segment(data = rbind(df_acf, df_pacf), aes(x = x, y = y, xend = xend, yend = yend), color = "gray30") +
  geom_point(data = rbind(df_acf, df_pacf), aes(x = x, y = y), color = "steelblue", size = 2) +
  
  # Intervalos de confianza para ACF y PACF
  geom_hline(data = df_ci, aes(yintercept = ci), linetype = "dashed", color = "firebrick") +
  geom_hline(data = rbind(df_acf, df_pacf), aes(yintercept = 0), color = "gray50") +
  
  # Ljung-Box (Puntos con color según p-valor)
  geom_point(data = df_lb, aes(x = x, y = y, color = grupo), size = 2.5) +
  geom_hline(data = df_lb, aes(yintercept = 0.05), linetype = "dashed", color = "firebrick") +
  scale_color_manual(values = c("No Rechazo" = "steelblue", "Rechazo" = "firebrick", "Normal Teórica" = "darkblue")) +
  
  # Histograma, Densidad Kernel y Curva Normal
  geom_histogram(data = df_hist, aes(x = x, y = after_stat(density)), bins = 20, fill = "lightblue", color = "white", alpha = 0.7) +
  geom_density(data = df_hist, aes(x = x), color = "darkred", linewidth = 0.8) +
  geom_line(data = df_norm, aes(x = x, y = y, color = grupo), linewidth = 0.8, linetype = "dashed") +
  
  # Organización en rejilla exclusiva de ggplot2
  facet_wrap(~ Panel, scales = "free", ncol = 2) +
  labs(
    title = "Diagnóstico Residual del Modelo SARIMA",
    x = "", y = "", color = ""
  ) +
  theme_minimal(base_size = 11) +
  theme(
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5),
    strip.text = element_text(face = "bold", size = 10),
    panel.grid.minor = element_blank(),
    legend.position = "bottom"
  )

<div style="text-align: justify; line-height: 1.5; font-size: 18px;"> 

### **Metodologías para Contrastar Normalidad**
</div>

---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;"> 

**Shapiro Wilk:** Prueba de Hipótesis para contrastar normalidad, la cual evalúa la normalidad analizando la correlación lineal entre los residuales ordenados y los valores teóricos esperados de una distribución normal.
</div>

In [ ]:
shapiro.test(res_sar)
if (shapiro.test(res_sar)$p.value < 0.05) {
  print("Los residuales no siguen una distribución normal")
} else {
  print("Los residuales siguen una distribución normal")
}

---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;"> 

**Kolmogorov-Smirnov:** Prueba no paramétrica basada en Bondad Ajuste, la cual compara la distribución acumulada empírica de los residuales con la función de distribución acumulada teórica normal.

</div>


In [ ]:
lillie.test(res_sar)
if (lillie.test(res_sar)$p.value < 0.05) {
  print("Los residuales no siguen una distribución normal")
} else {
  print("Los residuales siguen una distribución normal")
}

---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

**Jarque Bera:** Basado en la Asimetría y Curtosis de una Normal. Evalúa si la asimetría (S) y la curtosis (K) de los residuales coinciden conjuntamente con las de una distribución normal ($S=0$ y $K=3$).
</div>


In [ ]:
jbt <- jarqueberaTest(res_sar)
# Evaluar el p-value
if (jbt@test[["p.value"]] < 0.05) {
  print("Los residuales no siguen una distribución normal")
} else {
  print("Los residuales siguen una distribución normal")
}

In [ ]:
# Gráfico del Q-Q Plot #
qqPlot(res_sar, envelope = 0.95, main = "Q-Q Plot de Residuos con Intervalo")

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;"> 

### **Función *autoarima* - Modelos Preliminares**

La función *auto.arima()* de la librería forecast de R, proporciona una opción rápida para construir pronósticos con series temporales, debido a que evalúa entre todos los posibles modelos, al mejor modelo considerando diversos criterios: estacionariedad, estacionalidad, diferencias, entre otras.
</div>

In [ ]:
# Función Autoarima #
mod_auto <- auto.arima(sales)
mod_auto

In [ ]:
# Diagnóstico residual para auto.arima #
res_auto <- auto.arima(sales)$residuals
par(mfrow = c(2,2))
hist(res_auto)
shapiro.test(res_auto)
Box.test(res_auto, type = c("Box-Pierce", "Ljung-Box"))
Box.Test(res_auto)
acf(res_auto, main="ACF de Ventas")
pacf(res_auto, main="PACF de Ventas")

## Pronóstico

In [ ]:
# Modelo a ser empleado en el pronóstico #
mod_sar <- arima(sales, order = c(0, 1, 1), method = c("CSS-ML"))

In [ ]:
# Pronósticos para la serie de ventas con horizonte 30 (1 mes) #
fc_sar = forecast(sales, model = mod_sar, h = 12)
fc_sar

In [ ]:
# Datos de pronóstico
df <- data.frame(
  date = time(fc_sar$mean),
  forecast = as.numeric(fc_sar$mean),
  lower = as.numeric(fc_sar$lower[,2]),
  upper = as.numeric(fc_sar$upper[,2])
)

# Datos observados
df_obs <- data.frame(
  date = time(fc_sar$x),
  sales = as.numeric(fc_sar$x)
)

# Gráfico
ggplot() +
  geom_line(data = df_obs, aes(x = date, y = sales), color = "black", size = 1.2) +
  geom_ribbon(data = df, aes(x = date, ymin = lower, ymax = upper), fill = "skyblue", alpha = 0.4) +
  geom_line(data = df, aes(x = date, y = forecast), color = "blue", size = 1.2) +
  labs(title = "Pronósticos de las Ventas Promedio Mensual",
       x = "Mes", y = "Ventas Promedio") +
  theme_minimal() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 18, face = "bold")  # centrado y grande
  )

In [ ]:
# Forma Alternativa #
pred_sar <- predict(mod_sar, n.ahead = 12, se.fit = TRUE)
print(pred_sar)

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;"> 

### **Desempeño del Pronóstico**

Para medir el desempeño predictivo del modelo, se divide la serie en datos para entrenar (train) y datos para probar (test) el modelo. Tomando un 80% de los datos para ajustar el modelo y un 20% para testearlo, pues se sabe que entre más datos tenga el modelo podemos obtener mejores predicciones.

</div>

---
### **Analizar Desempeño por Trayectoria Ajustada**

In [ ]:
# 1. Ajustar el modelo ARIMA(0,1,1) a la serie 'sales'
mod_arima <- Arima(sales, order = c(0, 1, 1))

# 2. Obtener los valores ajustados (fitted values)
fitted_vals <- fitted(mod_arima)

# 3. Crear data frame para ggplot
# Se extraen los tiempos originales si 'sales' es un objeto de clase 'ts'
tiempo <- if (is.ts(sales)) as.numeric(time(sales)) else seq_along(sales)

df_fit <- data.frame(
  Tiempo = rep(tiempo, 2),
  Valor = c(as.numeric(sales), as.numeric(fitted_vals)),
  Serie = rep(c("Serie Original", "Ajustado ARIMA(0,1,1)"), each = length(sales))
)

# 4. Graficar la Serie Original vs. Trayectoria Ajustada
ggplot(df_fit, aes(x = Tiempo, y = Valor, color = Serie, linetype = Serie)) +
  geom_line(linewidth = 0.8) +
  scale_color_manual(values = c("Serie Original" = "steelblue", "Ajustado ARIMA(0,1,1)" = "firebrick")) +
  scale_linetype_manual(values = c("Serie Original" = "solid", "Ajustado ARIMA(0,1,1)" = "dashed")) +
  labs(
    title = "Serie de Ventas vs. Trayectoria Ajustada ARIMA(0,1,1)",
    x = "Tiempo",
    y = "Ventas",
    color = "",
    linetype = ""
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", hjust = 0.5),
    legend.position = "bottom",
    panel.grid.minor = element_blank()
  )

In [ ]:
# Con este comando se pueden ver las métricas para evaluar el pronóntico  ME, RMSE, MAE, MPE, MAPE, MASE #
print(summary(mod_sar))
# ME (Mean Error)
# RMSE (Root Mean Squared Error)
# MAE (Mean Absolute Error)
# MPE (Mean Percentage Error)
# MAPE (Mean Absolute Percentage Error)
# MASE (Mean Absolute Scaled Error)

---
### **Analizar Desempeño con Partición**

In [ ]:
# 1. Partición de los datos (70% Entrenamiento, 30% Prueba)
n <- length(sales)
n_train <- round(0.80 * n)
n_test <- n - n_train

# Extraer tiempos si es un objeto 'ts' o crear secuencia si es vector
tiempo_total <- if (is.ts(sales)) as.numeric(time(sales)) else seq_along(sales)

# Indexación directa (funciona para vectores, matrices y objetos ts)
sales_train <- sales[1:n_train]
sales_test  <- sales[(n_train + 1):n]

# Mantener la estructura 'ts' si el original era una serie de tiempo
if (is.ts(sales)) {
  sales_train <- ts(sales_train, frequency = frequency(sales), start = start(sales))
  sales_test  <- ts(sales_test, frequency = frequency(sales), start = time(sales)[n_train + 1])
}

tiempo_train <- tiempo_total[1:n_train]
tiempo_test  <- tiempo_total[(n_train + 1):n]

# 2. Ajustar ARIMA(0,1,1) sobre el 70% (Entrenamiento)
mod_train <- Arima(sales_train, order = c(0, 1, 1))

# 3. Pronosticar el 30% restante (Prueba)
fc_test <- forecast(mod_train, h = n_test)

# 4. Calcular el MAPE (Mean Absolute Percentage Error)
actual_test <- as.numeric(sales_test)
pred_test   <- as.numeric(fc_test$mean)

mape_val <- mean(abs((actual_test - pred_test) / actual_test)) * 100
mape_label <- sprintf("MAPE (Test): %.2f%%", mape_val)

# 5. Estructurar Data Frames para ggplot2
df_historico <- data.frame(
  Tiempo = c(tiempo_train, tiempo_test),
  Valor = c(as.numeric(sales_train), actual_test),
  Grupo = c(rep("Entrenamiento (70%)", n_train), rep("Prueba Real (30%)", n_test))
)

df_fc <- data.frame(
  Tiempo = tiempo_test,
  Pronostico = pred_test,
  Lower80 = as.numeric(fc_test$lower[, "80%"]),
  Upper80 = as.numeric(fc_test$upper[, "80%"]),
  Lower95 = as.numeric(fc_test$lower[, "95%"]),
  Upper95 = as.numeric(fc_test$upper[, "95%"])
)

# 6. Gráfica en ggplot2
ggplot() +
  # Bandas de confianza del pronóstico (80% y 95%)
  geom_ribbon(data = df_fc, aes(x = Tiempo, ymin = Lower95, ymax = Upper95), fill = "firebrick", alpha = 0.15) +
  geom_ribbon(data = df_fc, aes(x = Tiempo, ymin = Lower80, ymax = Upper80), fill = "firebrick", alpha = 0.25) +
  
  # Líneas de datos observados (Entrenamiento y Prueba)
  geom_line(data = df_historico, aes(x = Tiempo, y = Valor, color = Grupo), linewidth = 0.8) +
  
  # Línea del Pronóstico
  geom_line(data = df_fc, aes(x = Tiempo, y = Pronostico, color = "Pronóstico ARIMA(0,1,1)"), linewidth = 0.9, linetype = "dashed") +
  
  # Línea divisoria entre Entrenamiento y Prueba
  geom_vline(xintercept = tiempo_total[n_train], linetype = "dotted", color = "gray40", linewidth = 0.7) +
  
  # Anotación del MAPE en el gráfico
  annotate("label", x = min(tiempo_test), y = max(sales) * 0.98, 
           label = mape_label, fill = "white", color = "firebrick", fontface = "bold", size = 4, hjust = 0) +
  
  # Paletas y Estética
  scale_color_manual(
    values = c(
      "Entrenamiento (70%)" = "steelblue",
      "Prueba Real (30%)" = "black",
      "Pronóstico ARIMA(0,1,1)" = "firebrick"
    )
  ) +
  labs(
    title = "Evaluación Out-of-Sample: ARIMA(0,1,1)",
    subtitle = "Entrenamiento (70%) vs. Prueba (30%)",
    x = "Tiempo",
    y = "Ventas",
    color = ""
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", hjust = 0.5),
    plot.subtitle = element_text(hjust = 0.5),
    legend.position = "bottom",
    panel.grid.minor = element_blank()
  )

---
### **Analizar Desempeño con Validación Cruzada**

In [ ]:
# 1. Configuración de la Validación Cruzada con Ventana Expansiva
n <- length(sales)
k_folds <- 10

# Tamaño de la ventana inicial (por ejemplo, el 50% de la serie para comenzar)
min_train <- floor(n * 0.50)
# Paso o incremento de observaciones por cada fold
step <- floor((n - min_train) / k_folds)

mapes <- numeric(k_folds)

# 2. Bucle de Validación Cruzada (Expanding Window)
for (i in 1:k_folds) {
  # El punto de corte crece en cada iteración
  train_end <- min_train + (i - 1) * step
  test_end  <- min_train + i * step
  
  # Si es el último fold, incluir todas las observaciones restantes
  if (i == k_folds) test_end <- n
  
  # Partición de la serie
  train_fold <- sales[1:train_end]
  test_fold  <- sales[(train_end + 1):test_end]
  
  if (is.ts(sales)) {
    train_fold <- ts(train_fold, frequency = frequency(sales), start = start(sales))
    test_fold  <- ts(test_fold, frequency = frequency(sales), start = time(sales)[train_end + 1])
  }
  
  # Ajustar ARIMA(0,1,1) y pronosticar el bloque test
  h_horizon <- length(test_fold)
  fit_fold  <- Arima(train_fold, order = c(0, 1, 1))
  fc_fold   <- forecast(fit_fold, h = h_horizon)
  
  # Calcular MAPE del fold i
  actual_val <- as.numeric(test_fold)
  pred_val   <- as.numeric(fc_fold$mean)
  
  mapes[i] <- mean(abs((actual_val - pred_val) / actual_val)) * 100
}

# 3. Estadísticas Descriptivas de los MAPEs
mean_mape <- mean(mapes)
sd_mape   <- sd(mapes)

df_mapes <- data.frame(Fold = 1:k_folds, MAPE = mapes)

label_stats <- sprintf("Media: %.2f%%\nDesv. Est.: %.2f%%", mean_mape, sd_mape)

# 4. Histograma con ggplot2
ggplot(df_mapes, aes(x = MAPE)) +
  geom_histogram(bins = 6, fill = "lightblue", color = "white", alpha = 0.8) +
  
  # Línea de la Media
  geom_vline(xintercept = mean_mape, color = "firebrick", linetype = "dashed", linewidth = 1) +
  
  # Banda de +- 1 Desviación Estándar alrededor de la media
  annotate("rect", xmin = mean_mape - sd_mape, xmax = mean_mape + sd_mape, 
           ymin = -Inf, ymax = Inf, fill = "firebrick", alpha = 0.1) +
  
  # Anotación con Estadísticas
  annotate("label", x = max(mapes), y = Inf, label = label_stats, 
           vjust = 1.5, hjust = 1, fill = "white", color = "firebrick", fontface = "bold", size = 4) +
  
  labs(
    title = "Distribución de MAPEs (Validación Cruzada Expansiva - 10 Folds)",
    subtitle = "Modelo ARIMA(0,1,1)",
    x = "MAPE (%)",
    y = "Frecuencia / Número de Folds"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", hjust = 0.5),
    plot.subtitle = element_text(hjust = 0.5),
    panel.grid.minor = element_blank()
  )

<p align="center">
   <img src="img/cv_ts.jpeg" width="1200" height="600"> 
</p>

---
# 🎬 **¡FINAL!**
---